# 03 — Evaluate the Three Models on the Held-Out Test Set

Loads the three best checkpoints saved by `02_train_models.ipynb`, rebuilds
the **exact same test split** from `split_indices.json`, and produces every
evaluation artefact required by Bab IV:

- Confusion-matrix heatmaps (per model) — Master Coding Prompt §4.
- Per-class precision / recall / F1 + macro / weighted averages.
- Inference benchmark: mean / p50 / p95 ms per image (Master Prompt §4).
- A single tidy `final_comparison.csv` ready for Tabel 4.x.

Run on Colab GPU for fastest inference, but CPU also works (slower).

## 1 · Environment & Drive mount

In [2]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

if IN_COLAB:
    # Upgrade torch, torchvision, and torchaudio together to maintain consistency
    # and use newer scikit-learn/matplotlib to avoid dependency conflicts in newer Colab envs
    !pip -q install --upgrade torch==2.4.* torchvision==0.19.* torchaudio==2.4.* \
        "scikit-learn>=1.5" "seaborn>=0.13" "matplotlib>=3.9"

Running in Colab: True
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 114.7 MB/s eta 0:00:00


In [3]:
from pathlib import Path
import shutil, sys

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah')
    WORK_ROOT  = Path('/content/work')
else:
    DRIVE_ROOT = Path('..').resolve()
    WORK_ROOT  = DRIVE_ROOT

WORK_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'reports').mkdir(parents=True, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)

# Make utils.py importable
src_utils = DRIVE_ROOT / 'scripts' / 'utils.py'
if src_utils.exists():
    shutil.copy(src_utils, '/content/utils.py' if IN_COLAB else 'utils.py')
    sys.path.insert(0, '/content' if IN_COLAB else '.')
import utils, importlib
importlib.reload(utils)
from utils import (
    NUM_CLASSES, MODEL_NAMES, IMG_SIZE,
    seed_everything, get_transforms, build_model,
)

Mounted at /content/drive
DRIVE_ROOT = /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah


In [ ]:
# Make sure the dataset is unzipped on the Colab local disk.
import zipfile

DATA_ZIP  = DRIVE_ROOT / 'Dataset_Cropped.zip'
DATA_ROOT = WORK_ROOT

# Check if a representative subfolder of the dataset exists
if not (DATA_ROOT / 'defect').exists():
    if IN_COLAB:
        if not DATA_ZIP.exists():
            raise FileNotFoundError(
                f'Dataset zip not found.\n'
                f'Please upload Dataset_Cropped.zip to:\n  {DATA_ZIP}'
            )
        print(f'Extracting {DATA_ZIP} -> {WORK_ROOT} ...')
        with zipfile.ZipFile(DATA_ZIP) as zf:
            zf.extractall(WORK_ROOT)
        print('Done.')
    else:
        raise FileNotFoundError(
            f'Dataset folder not found at:\n  {DATA_ROOT}\n'
            f'Please run 01_auto_crop.py first to generate Dataset_Cropped/.'
        )
else:
    print(f'Dataset already present at {DATA_ROOT} — skipping extract.')

# Sanity check class folders
for c in ('defect', 'non_defect'):
    folder = DATA_ROOT / c
    if not folder.exists():
        raise FileNotFoundError(
            f'Expected class folder not found: {folder}\n'
            f'Dataset_Cropped/ must contain defect/ and non_defect/ subfolders.'
        )
    n = len(list(folder.glob('*.jpg')))
    print(f'  {c:>10}: {n:>5} images')

Dataset at /content/work/Dataset_Cropped


## 2 · Rebuild the **identical** test set

We do **not** call `random_split` again. Instead we replay the indices
saved by Script 2 against a fresh `ImageFolder` so the test set is
guaranteed-identical even if the working directory changed.

In [5]:
import json
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets

split_path = DRIVE_ROOT / 'reports' / 'split_indices.json'
if not split_path.exists():
    raise FileNotFoundError(
        f'{split_path} not found. Run 02_train_models.ipynb first.'
    )
split = json.loads(split_path.read_text())
CLASS_TO_IDX = split['class_to_idx']
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
CLASS_NAMES  = [IDX_TO_CLASS[i] for i in range(len(IDX_TO_CLASS))]
print('Classes:', CLASS_NAMES)

seed_everything(split['seed'])
full_dataset = datasets.ImageFolder(root=str(DATA_ROOT),
                                    transform=get_transforms(train=False))

# Sanity check: ImageFolder ordering must match what was saved.
saved_samples = split['samples']
current_samples = [s for s, _ in full_dataset.samples]
assert len(saved_samples) == len(current_samples), (
    f'Sample count differs from training run '
    f'({len(current_samples)} vs {len(saved_samples)}). '
    'Did the dataset change?'
)
# Compare only basenames so absolute-path differences across machines are OK.
saved_names   = [Path(p).name for p in saved_samples]
current_names = [Path(p).name for p in current_samples]
assert saved_names == current_names, (
    'ImageFolder ordering differs from training run. '
    'Re-zip the same Dataset_Cropped/ folder and retry.'
)

test_subset = Subset(full_dataset, split['test_indices'])
test_loader = DataLoader(test_subset, batch_size=64, shuffle=False,
                         num_workers=4,
                         pin_memory=torch.cuda.is_available())
print('Test images:', len(test_subset))

Classes: ['defect', 'non_defect']
Test images: 100


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


## 3 · Inference + metric helpers

In [6]:
import time
import numpy as np
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True  # auto-tune for fixed 224x224 input
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))


def evaluate_checkpoint(model_name: str) -> dict:
    ckpt_path = DRIVE_ROOT / 'checkpoints' / f'best_model_{model_name}.pth'
    if not ckpt_path.exists():
        raise FileNotFoundError(ckpt_path)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)

    model = build_model(model_name, pretrained=False)
    model.load_state_dict(ckpt['state_dict'])
    model.to(DEVICE).eval()

    y_true, y_pred, y_prob = [], [], []
    per_image_ms: list[float] = []

    # GPU warm-up so the first batch isn't penalised in the timing.
    if DEVICE.type == 'cuda':
        with torch.no_grad():
            dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
            _ = model(dummy)
            torch.cuda.synchronize()

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE, non_blocking=True)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            logits = model(x)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            elapsed_ms = (time.perf_counter() - t0) * 1000.0
            per_image_ms.extend([elapsed_ms / x.size(0)] * x.size(0))

            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
            y_true.extend(y.numpy().tolist())
            y_pred.extend(preds.tolist())
            y_prob.extend(probs.tolist())

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)

    acc = accuracy_score(y_true, y_pred)
    cm  = confusion_matrix(y_true, y_pred,
                           labels=list(range(len(CLASS_NAMES))))
    p, r, f, s = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(CLASS_NAMES))),
        zero_division=0,
    )
    report_dict = classification_report(
        y_true, y_pred, target_names=CLASS_NAMES,
        digits=4, zero_division=0, output_dict=True,
    )
    return {
        'model_name'  : model_name,
        'best_epoch'  : ckpt.get('best_epoch'),
        'best_val_acc': ckpt.get('best_val_acc'),
        'test_acc'    : acc,
        'cm'          : cm,
        'per_class_p' : p, 'per_class_r': r, 'per_class_f': f, 'support': s,
        'report_dict' : report_dict,
        'y_true'      : y_true, 'y_pred': y_pred, 'y_prob': y_prob,
        'ms_per_img'  : np.asarray(per_image_ms),
    }

Device: cuda
GPU   : Tesla T4


## 4 · Run inference for all three models

In [7]:
evaluations = {name: evaluate_checkpoint(name) for name in MODEL_NAMES}
for name, ev in evaluations.items():
    print(f'{name:>22s}  test_acc = {ev["test_acc"]:.4f}  '
          f'(best_val_acc={ev["best_val_acc"]:.4f}, ep={ev["best_epoch"]})')

/tmp/ipykernel_1328/3681345945.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=DEVICE)
/usr/local/lib/python3.12/dist-packages

    mobilenet_v3_large  test_acc = 0.9500  (best_val_acc=0.9700, ep=8)
              resnet50  test_acc = 0.9900  (best_val_acc=0.9800, ep=9)
                swin_t  test_acc = 0.9600  (best_val_acc=0.9800, ep=9)


## 5 · Confusion-matrix heatmaps (Gambar 4.7-style)

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns

for name, ev in evaluations.items():
    fig, ax = plt.subplots(figsize=(4.5, 4))
    sns.heatmap(
        ev['cm'], annot=True, fmt='d', cmap='Blues',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        cbar=False, ax=ax, square=True,
    )
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'{name} — test acc = {ev["test_acc"]:.4f}')
    fig.tight_layout()
    out = DRIVE_ROOT / 'reports' / f'confusion_matrix_{name}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved ->', out)

Saved -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/confusion_matrix_mobilenet_v3_large.png
Saved -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/confusion_matrix_resnet50.png
Saved -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/confusion_matrix_swin_t.png


## 6 · Per-class classification reports (CSV per model)

In [9]:
import pandas as pd

for name, ev in evaluations.items():
    rep_df = pd.DataFrame(ev['report_dict']).T
    rep_df = rep_df.round(4)
    out = DRIVE_ROOT / 'reports' / f'classification_report_{name}.csv'
    rep_df.to_csv(out, index=True)
    print(f'\n=== {name} ===')
    print(rep_df.to_string())
    print('Saved ->', out)


=== mobilenet_v3_large ===
              precision  recall  f1-score  support
defect           0.9574  0.9375    0.9474    48.00
non_defect       0.9434  0.9615    0.9524    52.00
accuracy         0.9500  0.9500    0.9500     0.95
macro avg        0.9504  0.9495    0.9499   100.00
weighted avg     0.9501  0.9500    0.9500   100.00
Saved -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/classification_report_mobilenet_v3_large.csv

=== resnet50 ===
              precision  recall  f1-score  support
defect           0.9796  1.0000    0.9897    48.00
non_defect       1.0000  0.9808    0.9903    52.00
accuracy         0.9900  0.9900    0.9900     0.99
macro avg        0.9898  0.9904    0.9900   100.00
weighted avg     0.9902  0.9900    0.9900   100.00
Saved -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/classification_report_resnet50.csv

=== swin_t ===
              precision  recall  f1-score  support
defect           0.9783  0.9375    0.9574    48.00
non_de

## 7 · Inference benchmark

In [10]:
bench_rows = []
for name, ev in evaluations.items():
    ms = ev['ms_per_img']
    bench_rows.append({
        'model'   : name,
        'device'  : DEVICE.type,
        'mean_ms' : round(float(np.mean(ms)), 3),
        'p50_ms'  : round(float(np.percentile(ms, 50)), 3),
        'p95_ms'  : round(float(np.percentile(ms, 95)), 3),
        'images'  : int(len(ms)),
    })
bench_df = pd.DataFrame(bench_rows)
bench_path = DRIVE_ROOT / 'reports' / 'inference_benchmark.csv'
bench_df.to_csv(bench_path, index=False)
print('Saved ->', bench_path)
bench_df

Saved -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/inference_benchmark.csv


,model,device,mean_ms,p50_ms,p95_ms,images
0,mobilenet_v3_large,cuda,6.509,5.909,7.574,100
1,resnet50,cuda,32.781,32.899,32.899,100
2,swin_t,cuda,4.896,5.079,5.079,100


## 8 · Final cross-model comparison table

This is the table that goes straight into Bab IV — one row per model with
every metric the examiners will ask about.

In [11]:
rows = []
for name, ev in evaluations.items():
    rep = ev['report_dict']
    rows.append({
        'model'             : name,
        'best_val_acc'      : round(float(ev['best_val_acc']), 4),
        'test_accuracy'     : round(float(ev['test_acc']), 4),
        'precision_defect'  : round(rep['defect']['precision'], 4),
        'recall_defect'     : round(rep['defect']['recall'],    4),
        'f1_defect'         : round(rep['defect']['f1-score'],  4),
        'precision_non_defect': round(rep['non_defect']['precision'], 4),
        'recall_non_defect' : round(rep['non_defect']['recall'],     4),
        'f1_non_defect'     : round(rep['non_defect']['f1-score'],   4),
        'macro_f1'          : round(rep['macro avg']['f1-score'],    4),
        'weighted_f1'       : round(rep['weighted avg']['f1-score'], 4),
        'mean_ms_per_image' : round(float(np.mean(ev['ms_per_img'])), 3),
    })
final_df = pd.DataFrame(rows)
final_path = DRIVE_ROOT / 'reports' / 'final_comparison.csv'
final_df.to_csv(final_path, index=False)
print('Saved ->', final_path)
final_df

Saved -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/final_comparison.csv


,model,best_val_acc,test_accuracy,precision_defect,recall_defect,f1_defect,precision_non_defect,recall_non_defect,f1_non_defect,macro_f1,weighted_f1,mean_ms_per_image
0,mobilenet_v3_large,0.97,0.95,0.9574,0.9375,0.9474,0.9434,0.9615,0.9524,0.9499,0.95,6.509
1,resnet50,0.98,0.99,0.9796,1.0000,0.9897,1.0000,0.9808,0.9903,0.9900,0.99,32.781
2,swin_t,0.98,0.96,0.9783,0.9375,0.9574,0.9444,0.9808,0.9623,0.9599,0.96,4.896


## 9 · Best-model declaration

Selection rule: highest **F1 on the `defect` class**. Defect is the
anomaly / positive class in a QC framing — a model that misses defects
(low recall on defect) is industrially worse than one that flags a few
extra non-defects.

In [12]:
best = final_df.sort_values('f1_defect', ascending=False).iloc[0]
print('=' * 60)
print('BEST MODEL (by F1 on defect class):', best['model'])
print('=' * 60)
for col in final_df.columns:
    print(f'  {col:>22s}: {best[col]}')

with open(DRIVE_ROOT / 'reports' / 'best_model.txt', 'w') as fp:
    fp.write(best.to_string())
print('\nSaved -> best_model.txt')

BEST MODEL (by F1 on defect class): resnet50
                   model: resnet50
            best_val_acc: 0.98
           test_accuracy: 0.99
        precision_defect: 0.9796
           recall_defect: 1.0
               f1_defect: 0.9897
    precision_non_defect: 1.0
       recall_non_defect: 0.9808
           f1_non_defect: 0.9903
                macro_f1: 0.99
             weighted_f1: 0.99
       mean_ms_per_image: 32.781

Saved -> best_model.txt


✅ **Phase 3 done.** All evaluation artefacts are in `MyDrive/TA/reports/`:

- `confusion_matrix_<model>.png` — Gambar 4.7
- `classification_report_<model>.csv` — Tabel 4.x per model
- `inference_benchmark.csv` — speed comparison
- `final_comparison.csv` — single-table summary for Bab IV
- `best_model.txt` — declared winner

When examiners ask deeper questions, see
`../Skripsi_Split_MD/15_PENGEMBANGAN_MENDATANG.md` for the parked
Stratified-split, Grad-CAM, CPU-vs-GPU, k-fold, multi-seed, and
edge-deployment extensions.